# PC-TTP Alignment Hypothesis Validation

Cheap validation pass for `20260813-pc_ttp_anchored_curve_resampler_v2.md`, **before**
touching `04_cross_dataset_training.py`/`combine_group` for real: does aligning chips
by their PC well's time-to-positivity (instead of raw acquisition-start zeroing) make
the *same target* actually converge across chips? If PC itself doesn't converge after
alignment, timestamp misalignment isn't (or isn't the whole) story.

This notebook does **not** modify any existing `.py` file — it imports
`04_cross_dataset_training.py` (`combine_group`'s sibling helpers, `CurveResampler`)
and `config.py` as-is, and implements the new PC-TTP alignment logic entirely here,
matching `combine_group`'s exact output shape so the existing `plot_target_across_chips`
(`lofo_class_balance_analysis.ipynb` §10) can be reused unchanged on the result.

**Decisions already made** (per discussion, not re-litigated here):
- Anchor is **fold-scoped**: computed from every chip *except* whichever one is being
  treated as "held out" for that comparison, then applied (with graceful clipping) to
  every chip including the held-out one — this is also how we test the "unseen chip
  has an earlier TTP than the anchor" edge case for real, not hypothetically.
- Both `anchor_method="min"` and `anchor_method="percentile"` are implemented as a
  parameter, to compare visually rather than assume one is better.
- This is a **validation-only** notebook. Whether this becomes a real opt-in mode in
  `04_cross_dataset_training.py` is a separate decision, made after inspecting these
  results — nothing here is wired into the production pipeline.

In [1]:
import sys
import importlib
import colorsys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
import os

try:
    # VS Code injects '__vsc_ipynb_file__' into the globals automatically
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(os.path.dirname(notebook_path))
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config
import joblib

# Import-only, per instructions -- 04_cross_dataset_training.py itself is never
# modified. cdt's own imports already put utils/model_training on sys.path.
cdt = importlib.import_module("04_cross_dataset_training")
from model_utils import CurveResampler   # same resampler class combine_group uses
import sigmoid_fitting as sp             # for computing PC's Ct fresh -- see §2

%matplotlib inline


Current Working Directory: /vol/bitbucket/gk225/POC_DDM/gk_code/main

[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



## 1. Configuration

In [3]:
GROUP_NAME = "final_6_chip_clean_nn"
EXP_FOLDER = config.FINAL_EXP_FOLDER + "_nc_subtract"
CURVE_TYPE = "ori_curve_sg_p4_norm"

folder_names = config.CROSS_DATASET_GROUPS[GROUP_NAME]
exp_paths = [Path(EXP_FOLDER, name) for name in folder_names]

def short_name(folder):
    return folder.split('_U_', 1)[1]

print(f"Group '{GROUP_NAME}' -> {len(folder_names)} chips:")
for n in folder_names:
    print(" -", n, f"({short_name(n)})")


Group 'final_6_chip_clean_nn' -> 6 chips:
 - D20260806_E00_C00_F4500KHz_U_DDM_01_06 (DDM_01_06)
 - D20260807_E00_C00_F4500KHz_U_DDM_02_07 (DDM_02_07)
 - D20260808_E00_C00_F4500KHz_U_DDM_03_01 (DDM_03_01)
 - D20260810_E00_C00_F4500KHz_U_DDM_04_01 (DDM_04_01)
 - D20260825_E00_C00_F4500KHz_U_DDM_05_01 (DDM_05_01)
 - D20260825_E00_C00_F4500KHz_U_DDM_06_02 (DDM_06_02)


## 2. Load PC ground truth + compute TTP per chip

In [ ]:
from joblib import Memory

CACHE_DIR = Path(EXP_FOLDER) / "cross_dataset_cv" / GROUP_NAME / "_cache_pc_ttp_validation"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
memory = Memory(str(CACHE_DIR), verbose=0)


def load_pc_wells_snapshot(exp_path, curve_type):
    """Reads 01's --drop_pc PC snapshot directly from curve_for_training.joblib's
    'pc_wells' key -- PC never reaches load_curve_data()'s main arrays, since
    --drop_pc removes it at the preprocessing stage before 02/03/04 ever see it
    (confirmed: this group's joblib has Y_well values [0,1,2,3,4,5,6,9] only --
    well 8/PC is absent -- and a separate 'pc_wells' key with 791 PC pixels).
    No kinetic_features/Ct here (the snapshot skips sigmoid fitting,
    compute_sigmoid_fits=False in 01_curve_preprocessing_v6.py)."""
    data_path = os.path.join(exp_path, config.TRAINING_DATA_PATH)
    if not os.path.exists(data_path):
        return None
    data = joblib.load(data_path)
    pc = data.get("pc_wells")
    if not pc:
        print(f"  [!] {exp_path.name}: no 'pc_wells' snapshot (--drop_pc not used for this experiment?).")
        return None
    resolved = config.CURVE_TYPE_ALIASES.get(curve_type, curve_type)
    if resolved not in pc["curves"]:
        print(f"  [!] {exp_path.name}: PC snapshot has no '{resolved}' variant "
              f"(available: {list(pc['curves'].keys())}).")
        return None
    return {
        "curves": np.asarray(pc["curves"][resolved]),
        "timestamps": np.asarray(data["timestamps"], dtype=float),
    }


def compute_ct_for_curves(curves, timestamps):
    """Mean Ct across a curve batch, fit fresh -- 01's --drop_pc snapshot skips
    kinetic-feature extraction, so PC has no precomputed Ct to reuse. Same
    underlying function 02_outlier_detection_pipeline.py uses for every other well."""
    cts = []
    for y in curves:
        valid = np.isfinite(timestamps) & np.isfinite(y)
        if valid.sum() < 3:
            continue
        try:
            feats = sp.extract_kinetic_parameters_original(timestamps, y)
            if "Ct" in feats and np.isfinite(feats["Ct"]):
                cts.append(feats["Ct"])
        except Exception:
            continue
    return float(np.mean(cts)) if cts else None


@memory.cache
def _pc_ttp_for_chip_cached(exp_path_str, curve_type):
    """Disk-cached under CACHE_DIR -- sigmoid-fitting ~800 PC pixels per chip takes
    minutes (confirmed: >120s for one chip), and the result is deterministic given
    (chip, curve_type), so it only needs computing once. joblib.Memory keys on this
    function's own source too, so editing load_pc_wells_snapshot/compute_ct_for_curves
    correctly invalidates stale entries; delete CACHE_DIR to force a full refit
    regardless (e.g. if 01's underlying PC snapshot data changes)."""
    exp_path = Path(exp_path_str)
    pc = load_pc_wells_snapshot(exp_path, curve_type)
    if pc is None:
        return None
    return compute_ct_for_curves(pc["curves"], pc["timestamps"])


def pc_ttp_per_chip(exp_paths, curve_type):
    """{chip_name: mean Ct over that chip's PC snapshot pixels}."""
    ttp = {}
    for exp_path in exp_paths:
        ct = _pc_ttp_for_chip_cached(str(exp_path), curve_type)
        if ct is None:
            print(f"  [!] {exp_path.name}: no PC TTP available for curve_type={curve_type!r} "
                  f"(see load_pc_wells_snapshot/compute_ct_for_curves warnings above, if any).")
            continue
        ttp[exp_path.name] = ct
    return ttp


pc_ttp = pc_ttp_per_chip(exp_paths, CURVE_TYPE)
print("\nPC TTP (mean Ct) per chip:")
for name, v in sorted(pc_ttp.items(), key=lambda kv: kv[1]):
    print(f"  {short_name(name):>12s}: {v:.2f}")
print(f"\nSpread: {max(pc_ttp.values()) - min(pc_ttp.values()):.2f} "
      f"(min={min(pc_ttp.values()):.2f}, max={max(pc_ttp.values()):.2f})")
print("If this spread is tiny relative to the curve duration, timestamp misalignment "
      "probably isn't a meaningful factor here -- worth checking before reading too "
      "much into the figures below.")


/vol/bitbucket/gk225/POC_DDM/gk_code/main/utils/sigmoid_fitting.py:27: RuntimeWarning: overflow encountered in power
  denominator = (1.0 + exp_term)**(As + 1)
/vol/bitbucket/gk225/POC_DDM/gk_code/main/utils/sigmoid_fitting.py:27: RuntimeWarning: overflow encountered in power
  denominator = (1.0 + exp_term)**(As + 1)


## 3. PC-TTP-aligned `combine_group` (fold-scoped, parameterized anchor)

In [ ]:
def merge_pc_into_part(part, exp_path, curve_type):
    pc = load_pc_wells_snapshot(exp_path, curve_type)
    if pc is None:
        return part
    if pc["curves"].shape[1] != part["curves"].shape[1]:
        print(f"  [!] {exp_path.name}: PC has {pc['curves'].shape[1]} timepoints, "
              f"main data has {part['curves'].shape[1]} -- skipping PC merge.")
        return part
    n_pc = pc["curves"].shape[0]
    part = dict(part)
    part["curves"] = np.concatenate([part["curves"], pc["curves"]], axis=0)
    part["Y_mapped"] = np.concatenate([part["Y_mapped"], np.full(n_pc, "PC", dtype=object)])
    if part.get("well_ids") is not None:
        part["well_ids"] = np.concatenate([part["well_ids"], np.full(n_pc, f"{exp_path.name}::8", dtype=object)])
    if part.get("concentration_raw") is not None:
        part["concentration_raw"] = np.concatenate([
            np.asarray(part["concentration_raw"], dtype=object), np.full(n_pc, None, dtype=object)])
    return part


def combine_group_pc_aligned(exp_paths, curve_type, held_out_chip=None,
                             anchor_method="min", anchor_pct=10, pc_ttp=None, verbose=True):
    parts = []
    for exp_path in exp_paths:
        d = cdt.load_curve_data(exp_path, curve_type, group_name=GROUP_NAME)   # LOFO_EXCLUDE_WELL_MAPPING[GROUP_NAME] (Cov/Hadv etc., as real training sees it)
        if d is None:
            continue
        d = merge_pc_into_part(d, exp_path, curve_type)  # PC added back in just for this diagnostic
        parts.append(d)
    if len(parts) < 2:
        print("  -> fewer than 2 usable chips.")
        return None

    pc_ttp_local = pc_ttp if pc_ttp is not None else pc_ttp_per_chip(exp_paths, curve_type)

    train_ttps = [v for k, v in pc_ttp_local.items() if k != held_out_chip]
    if not train_ttps:
        raise ValueError("No PC TTP available for any training chip.")
    if anchor_method == "min":
        anchor = min(train_ttps)
    elif anchor_method == "percentile":
        anchor = float(np.percentile(train_ttps, anchor_pct))
    else:
        raise ValueError(f"Unknown anchor_method: {anchor_method!r}")

    # --- front truncation: align each chip's PC TTP to the anchor (clip, never fail) ---
    aligned = []
    shifts = {}
    for p in parts:
        ttp = pc_ttp_local.get(p["dataset_id"])
        shift = max(ttp - anchor, 0.0) if ttp is not None else 0.0
        shifts[p["dataset_id"]] = shift
        start_idx = int(np.searchsorted(p["timestamps"], p["timestamps"][0] + shift))
        start_idx = min(start_idx, len(p["timestamps"]) - 1)
        p2 = dict(p)
        p2["timestamps"] = p["timestamps"][start_idx:]
        p2["curves"] = p["curves"][:, start_idx:]
        aligned.append(p2)

    # --- back truncation: common duration from training chips only, clip for others ---
    train_lens = [p["timestamps"][-1] - p["timestamps"][0]
                  for p in aligned if p["dataset_id"] != held_out_chip]
    common_duration = min(train_lens)
    for p2 in aligned:
        end_time = p2["timestamps"][0] + common_duration
        end_idx = min(int(np.searchsorted(p2["timestamps"], end_time)) + 1, len(p2["timestamps"]))
        p2["timestamps"] = p2["timestamps"][:end_idx]
        p2["curves"] = p2["curves"][:, :end_idx]

    if verbose:
        print(f"  anchor ({anchor_method}) = {anchor:.2f}  |  held_out = {short_name(held_out_chip) if held_out_chip else None}")
        for name, s in shifts.items():
            flag = " <- CLIPPED (TTP below anchor)" if pc_ttp_local.get(name, anchor) < anchor else ""
            print(f"    {short_name(name):>12s}: shift={s:8.2f}{flag}")
        print(f"  common_duration = {common_duration:.2f}")

    # --- resample onto one common grid: SAME CurveResampler cdt.combine_group uses ---
    timestamps_zeroed = [p["timestamps"] - p["timestamps"][0] for p in aligned]
    resampler = CurveResampler.fit(timestamps_zeroed)
    for p in aligned:
        p["curves"] = resampler.transform(p["timestamps"], p["curves"])

    conc_parts = []
    for p in aligned:
        raw = p.get("concentration_raw")
        conc_parts.append(np.asarray(raw, dtype=object) if raw is not None
                          else np.full(len(p["Y_mapped"]), None, dtype=object))

    combined = {
        "curves": np.concatenate([p["curves"] for p in aligned], axis=0),
        "Y_mapped": np.concatenate([p["Y_mapped"] for p in aligned], axis=0),
        "dataset_id": np.concatenate([np.full(len(p["Y_mapped"]), p["dataset_id"], dtype=object) for p in aligned], axis=0),
        "dataset_names": [p["dataset_id"] for p in aligned],
        "resampler": resampler,
        "coords": None,   # PC rows have no coords; unused by plot_target_across_chips/build_well_table
        "well_ids": (np.concatenate([p["well_ids"] for p in aligned], axis=0)
                     if all(p.get("well_ids") is not None for p in aligned) else None),
        "concentration_raw": np.concatenate(conc_parts, axis=0),
    }
    return combined, pc_ttp_local, shifts, anchor


## 4. Visual check 1 — does the same target converge across chips after alignment?

In [ ]:
def build_well_table(combined):
    df = pd.DataFrame({
        "well_id": combined["well_ids"],
        "dataset_id": combined["dataset_id"],
        "label": combined["Y_mapped"],
    })
    df["row_idx"] = np.arange(len(df))
    meta = df.groupby("well_id").agg(dataset_id=("dataset_id", "first"), label=("label", "first"))
    row_idx_map = df.groupby("well_id")["row_idx"].apply(np.array)
    wells = meta.join(row_idx_map.rename("row_idx")).reset_index()
    wells["n_pixels"] = wells["row_idx"].apply(len)
    return wells


def _shade(base_rgb, frac, spread=0.3):
    h, l, s = colorsys.rgb_to_hls(*base_rgb[:3])
    l = min(0.85, max(0.15, l + (frac - 0.5) * spread))
    return colorsys.hls_to_rgb(h, l, s)


def _exp_colors(exp_names):
    cmap_exp = plt.cm.tab10
    return {n: cmap_exp(i % 10) for i, n in enumerate(exp_names)}


def _exp_legend(fig, exp_color, exp_names, y_anchor=-0.02):
    handles = [plt.Line2D([0], [0], color=exp_color[n], lw=2, label=short_name(n))
               for n in exp_names]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(exp_names), 4),
               fontsize=8, bbox_to_anchor=(0.5, y_anchor), frameon=True)


def _raw_conc(combined, well_row):
    conc = combined.get("concentration_raw")
    if conc is None:
        return None
    val = conc[well_row["row_idx"][0]]
    try:
        return float(val)
    except (TypeError, ValueError):
        return None


def plot_target_across_chips(combined, label_order=None, figsize_per_cell=(6, 4),
                             title=None, alpha_indiv=0.10, with_all_curves=True,
                             sharey=False, y_scale=None, non_amp_filter=False):
    """Copied from lofo_class_balance_analysis.ipynb §10, plus non_amp_filter: drop
    individual pixel curves whose last value < start value ("non-amplified") before
    computing each well's mean/plotted curves, printing how many were removed per
    (exp, well, label, concentration)."""
    wells = build_well_table(combined)
    exp_names = sorted(wells["dataset_id"].unique())
    exp_color = _exp_colors(exp_names)

    labels_seen = list(dict.fromkeys(wells.sort_values("dataset_id")["label"]))
    if label_order is not None:
        all_labels = [l for l in label_order if l in labels_seen]
        all_labels += [l for l in labels_seen if l not in all_labels]
    else:
        nc    = [l for l in labels_seen if str(l).startswith("NC")]
        nonnc = [l for l in labels_seen if not str(l).startswith("NC")]
        all_labels = sorted(nonnc) + sorted(nc)

    t = np.asarray(combined["resampler"].t_grid, dtype=float)
    n = len(all_labels)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per_cell[0] * n, figsize_per_cell[1]),
                             squeeze=False, sharey="row" if sharey else False)
    axes = axes[0]

    for ax, label in zip(axes, all_labels):
        label_wells = wells[wells["label"] == label].copy()
        label_wells["_conc"] = [_raw_conc(combined, w) for _, w in label_wells.iterrows()]
        concs_sorted = sorted({c for c in label_wells["_conc"] if c is not None})
        n_concs = len(concs_sorted)

        for _, w in label_wells.iterrows():
            idx = w["row_idx"]
            curves = combined["curves"][idx]
            conc = w["_conc"]

            if non_amp_filter:
                keep = curves[:, -1] >= curves[:, 0]
                n_removed = int((~keep).sum())
                if n_removed:
                    print(f"  [non_amp_filter] {w['dataset_id']} | well={w['well_id']} | "
                          f"label={label} | conc={conc} | removed {n_removed}/{len(curves)} "
                          f"non-amplified curves")
                curves = curves[keep]
                if len(curves) == 0:
                    continue

            mean = curves.mean(axis=0)
            frac = (concs_sorted.index(conc) / (n_concs - 1)) if (conc is not None and n_concs > 1) else 0.5
            # color = _shade(exp_color[w["dataset_id"]], 1 - frac)
            color = _shade(exp_color[w["dataset_id"]], 1)

            if with_all_curves:
                step = max(1, len(curves) // 30)
                for c in curves[::step]:
                    ax.plot(t, c, color=color, lw=0.4, alpha=alpha_indiv, rasterized=True)
            ax.plot(t, mean, color=color, lw=1.5)

        ax.set_title(str(label), fontsize=11, fontweight="bold")
        ax.set_xlabel("Time", fontsize=8)
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.tick_params(labelsize=7)
        if y_scale is not None:
            ax.set_ylim(*y_scale)

    axes[0].set_ylabel("Signal", fontsize=9)
    fig.suptitle(title or "Same target across chips  (colour=chip, shade=concentration)",
                fontsize=12, fontweight="bold")
    _exp_legend(fig, exp_color, exp_names)
    plt.tight_layout(rect=[0, 0.08, 1, 0.93])
    plt.show()


print("Visualisation helpers defined.")


### 4a. Baseline — current production alignment (no PC-TTP), for comparison

In [ ]:
combined_baseline = cdt.combine_group(exp_paths, GROUP_NAME, curve_type=CURVE_TYPE)
plot_target_across_chips(
    combined_baseline,
    title=f"BASELINE (current production, no PC-TTP alignment) | {GROUP_NAME} | {CURVE_TYPE}",
    with_all_curves=False,
)


### 4b. PC-TTP-aligned — pick one (held-out chip, anchor method) to inspect

In [ ]:
HELD_OUT_CHIP = folder_names[0]   # e.g. treat the first chip as "unseen"
ANCHOR_METHOD = "min"             # "min" | "percentile"
ANCHOR_PCT    = 10

result = combine_group_pc_aligned(exp_paths, CURVE_TYPE, held_out_chip=HELD_OUT_CHIP,
                                  anchor_method=ANCHOR_METHOD, anchor_pct=ANCHOR_PCT,
                                  pc_ttp=pc_ttp)
combined_aligned, pc_ttp_local, shifts, anchor = result

plot_target_across_chips(
    combined_aligned,
    title=(f"PC-TTP ALIGNED | held_out={short_name(HELD_OUT_CHIP)} | "
          f"anchor={ANCHOR_METHOD} | {GROUP_NAME} | {CURVE_TYPE}"),
    with_all_curves=False,
)


### 4c. Full sweep — every chip held out, both anchor methods

In [ ]:
for held_out in folder_names:
    for method in ["min", "percentile"]:
        print(f"\n=== held_out={short_name(held_out)}  anchor={method} ===")
        result = combine_group_pc_aligned(exp_paths, CURVE_TYPE, held_out_chip=held_out,
                                          anchor_method=method, anchor_pct=ANCHOR_PCT,
                                          pc_ttp=pc_ttp)
        if result is None:
            continue
        combined_i, _, _, _ = result
        plot_target_across_chips(
            combined_i,
            title=f"PC-TTP ALIGNED | held_out={short_name(held_out)} | anchor={method}",
            with_all_curves=False,
        )


## 5. Visual check 2 — per-chip before/after (resampler-visualization style)

In [ ]:
def plot_pc_alignment_effect_for_exp(exp_path, combined_before, combined_after,
                                     before_label="baseline", after_label="PC-TTP aligned"):
    mask_before = combined_before["dataset_id"] == exp_path.name
    mask_after  = combined_after["dataset_id"] == exp_path.name

    wells_before = build_well_table(combined_before)
    wells_before = wells_before[wells_before["dataset_id"] == exp_path.name]
    wells_after  = build_well_table(combined_after)
    wells_after  = wells_after[wells_after["dataset_id"] == exp_path.name]

    t_before = np.asarray(combined_before["resampler"].t_grid, dtype=float)
    t_after  = np.asarray(combined_after["resampler"].t_grid, dtype=float)
    x_before = (t_before - t_before[0]) / (t_before[-1] - t_before[0])
    x_after  = (t_after - t_after[0]) / (t_after[-1] - t_after[0])

    wells = sorted(set(wells_before["well_id"]) | set(wells_after["well_id"]))
    ncols = min(5, len(wells))
    nrows = int(np.ceil(len(wells) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows), squeeze=False)

    for i, well in enumerate(wells):
        ax = axes[i // ncols, i % ncols]
        wb = wells_before[wells_before["well_id"] == well]
        wa = wells_after[wells_after["well_id"] == well]
        label = wb.iloc[0]["label"] if len(wb) else wa.iloc[0]["label"]

        if len(wb):
            mean_b = combined_before["curves"][wb.iloc[0]["row_idx"]].mean(axis=0)
            ax.plot(x_before, mean_b, color="#2a78d6", lw=1.4, label=before_label)
        if len(wa):
            mean_a = combined_after["curves"][wa.iloc[0]["row_idx"]].mean(axis=0)
            ax.plot(x_after, mean_a, color="#eb6834", lw=1.4, ls="--", label=after_label)

        ax.set_title(f"well {well.split('::')[-1]} ({label})", fontsize=9)
        ax.set_xlabel("Normalised time", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, loc="upper left")
        ax.grid(alpha=0.3)

    for j in range(len(wells), nrows * ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.suptitle(f"PC-TTP alignment effect — {exp_path.name}", fontsize=12, fontweight="bold")
    fig.tight_layout()
    plt.show()


for exp_path in exp_paths:
    plot_pc_alignment_effect_for_exp(exp_path, combined_baseline, combined_aligned)


## 6. Normalize before vs. after truncation

Per-curve min-max normalization (`normalize_curves_minmax`, copied verbatim from
`01_curve_preprocessing_v6.py`) can be applied at two different points relative to the
PC-TTP alignment:

- **Before** (current production order): `01` already normalizes the full, untrimmed
  curve — `combine_group_pc_aligned(..., curve_type="ori_curve_norm")` picks up that
  pre-normalized variant, so alignment/truncation just windows into an already-`[0,1]`
  -scaled curve. Stable, curve-wide amplitude reference, but the aligned *window*
  (what the classifier actually sees) may occupy a different sub-range of `[0,1]` per
  chip.
- **After**: align/truncate/resample the raw curve first, then normalize the
  resulting window (`normalize_combined_curves`). Guarantees every chip's aligned
  window spans the full `[0,1]` range, at the risk of stretching noise to fill that
  range for weak-signal curves (NC, low concentration) whose windowed segment doesn't
  capture much true dynamic range.

PC's own TTP is unaffected by which variant is used here — min-max normalization is a
strictly monotonic affine transform of the curve's y-values, so it doesn't shift
*where* a percentage-of-range crossing happens in time, only the curve's amplitude
scale. `pc_ttp` from §2 (computed on raw `ori_curve`) is reused for both variants
below rather than refitting.

In [ ]:
def normalize_curves_minmax(curves):
    """Copied verbatim from 01_curve_preprocessing_v6.py -- per-curve min-max to [0,1]."""
    curves = np.asarray(curves, dtype=np.float64)
    row_min = curves.min(axis=1, keepdims=True)
    row_max = curves.max(axis=1, keepdims=True)
    denom = np.where(row_max - row_min == 0, 1, row_max - row_min)
    return (curves - row_min) / denom


def normalize_combined_curves(combined):
    """Copy of `combined` with curves per-curve min-max normalized -- apply to an
    ALREADY-built combined dict (i.e. post-truncation/resampling) for the
    "normalize-after" variant."""
    out = dict(combined)
    out["curves"] = normalize_curves_minmax(combined["curves"])
    return out


print("normalize_curves_minmax / normalize_combined_curves defined.")


In [ ]:
# --- normalize-BEFORE: 01's pre-existing "ori_curve_norm" variant (normalized before any truncation) ---
result_before = combine_group_pc_aligned(exp_paths, "ori_curve_wavelet_bior35_norm", held_out_chip=HELD_OUT_CHIP,
                                         anchor_method=ANCHOR_METHOD, anchor_pct=ANCHOR_PCT,
                                         pc_ttp=pc_ttp)
combined_norm_before, _, _, _ = result_before
plot_target_across_chips(
    combined_norm_before,
    title=f"NORMALIZE-BEFORE truncation | held_out={short_name(HELD_OUT_CHIP)} | anchor={ANCHOR_METHOD}",
    with_all_curves=False,
)


In [ ]:
# --- normalize-AFTER: raw curves aligned/truncated/resampled first, then normalized ---
result_after = combine_group_pc_aligned(exp_paths, "ori_curve_wavelet_bior35_norm", held_out_chip=HELD_OUT_CHIP,
                                        anchor_method=ANCHOR_METHOD, anchor_pct=ANCHOR_PCT,
                                        pc_ttp=pc_ttp)
combined_raw_aligned, _, _, _ = result_after
combined_norm_after = normalize_combined_curves(combined_raw_aligned)
plot_target_across_chips(
    combined_norm_after,
    title=f"NORMALIZE-AFTER truncation | held_out={short_name(HELD_OUT_CHIP)} | anchor={ANCHOR_METHOD}",
    with_all_curves=False,
)


In [ ]:
# --- NORMALIZE-AFTER truncation, with non-amplified curves removed ---
plot_target_across_chips(
    combined_norm_after,
    title=(f"NORMALIZE-AFTER truncation | non-amplified removed | "
          f"held_out={short_name(HELD_OUT_CHIP)} | anchor={ANCHOR_METHOD}"),
    with_all_curves=False,
    non_amp_filter=True,
)


### 6a. Focused check — weakest-signal label (NC) under both variants

Where the "stretch noise to fill [0,1]" risk from normalize-after would show up
first, if it's real.

In [ ]:
def plot_label_before_after(combined_before, combined_after, label,
                            before_label="normalize-before", after_label="normalize-after"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, combined, title in zip(axes, [combined_before, combined_after],
                                   [before_label, after_label]):
        wells = build_well_table(combined)
        label_wells = wells[wells["label"] == label]
        t = np.asarray(combined["resampler"].t_grid, dtype=float)
        exp_names = sorted(wells["dataset_id"].unique())
        exp_color = _exp_colors(exp_names)
        for _, w in label_wells.iterrows():
            mean = combined["curves"][w["row_idx"]].mean(axis=0)
            ax.plot(t, mean, color=exp_color[w["dataset_id"]], lw=1.5, label=short_name(w["dataset_id"]))
        ax.set_title(f"{label} — {title}", fontsize=10, fontweight="bold")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    fig.suptitle(f"'{label}' before vs. after normalization order", fontsize=12, fontweight="bold")
    fig.tight_layout()
    plt.show()


plot_label_before_after(combined_norm_before, combined_norm_after, label="NC")


## 7. First derivative of PC-TTP-aligned curves

§4/§6 above compare curve *amplitude* after alignment (and after normalize-before vs.
normalize-after). Amplitude can still differ chip-to-chip even post-normalization if
the aligned window lands on a different sub-range of each chip's dynamic range. The
1st derivative (`d(signal)/dt`) is a different lens: it highlights *where* each curve
is rising/falling fastest, which is closer to true kinetics and less sensitive to a
chip-specific vertical offset.

Taken **after** `combine_group_pc_aligned` (+ whichever normalization variant), on the
resampler's common time grid (`np.gradient`, central differences) -- so alignment,
truncation, resampling, and normalization order are all unaffected; only the curve
being plotted is replaced by its derivative.

Two variants, mirroring §6:
- **1st derivative of normalize-before**: `differentiate_combined_curves(combined_norm_before)`
- **1st derivative of normalize-after**: `differentiate_combined_curves(combined_norm_after)`

In [ ]:
def differentiate_combined_curves(combined):
    """Copy of `combined` with curves replaced by their 1st derivative w.r.t.
    resampler.t_grid (np.gradient, central differences). Apply AFTER
    combine_group_pc_aligned (+ optionally normalize_combined_curves) -- the
    derivative is taken on the already-aligned/truncated/resampled(/normalized)
    curve, not the raw one."""
    t = np.asarray(combined["resampler"].t_grid, dtype=float)
    curves = np.asarray(combined["curves"], dtype=np.float64)
    out = dict(combined)
    out["curves"] = np.gradient(curves, t, axis=1)
    return out


print("differentiate_combined_curves defined.")

In [ ]:
# --- 1st derivative of NORMALIZE-BEFORE ---
combined_deriv_before = differentiate_combined_curves(combined_norm_before)
plot_target_across_chips(
    combined_deriv_before,
    title=(f"1st DERIVATIVE | NORMALIZE-BEFORE truncation | held_out={short_name(HELD_OUT_CHIP)} | "
          f"anchor={ANCHOR_METHOD}"),
    with_all_curves=False,
)

In [ ]:
# --- 1st derivative of NORMALIZE-AFTER ---
combined_deriv_after = normalize_combined_curves(combined_deriv_before)
plot_target_across_chips(
    combined_deriv_after,
    title=(f"1st DERIVATIVE | NORMALIZE-AFTER truncation | held_out={short_name(HELD_OUT_CHIP)} | "
          f"anchor={ANCHOR_METHOD}"),
    with_all_curves=False,
)

### 7a. Focused check — weakest-signal label (NC), derivative before vs. after

Same NC focus as §6a, on the 1st-derivative curves instead of amplitude.

In [ ]:
plot_label_before_after(
    combined_deriv_before, combined_deriv_after, label="NC",
    before_label="1st deriv (normalize-before)", after_label="1st deriv (normalize-after)",
)